Install and import the required libraries

In [ ]:
import numpy as np
import pandas as pd
import os

I. Data cleaning and preparation

1. Oustanding Shares

In [ ]:
df = pd.read_excel("annual_shares_outstanding.xlsx")

col_shares = "Outstanding Share (Mil. Shares)"

# Standardize columns
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df[col_shares] = pd.to_numeric(df[col_shares], errors="coerce")

# Drop invalid keys
df = df.dropna(subset=["ticker", "Year"]).copy()

# Sort before time-series operations
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

df = df.drop_duplicates(subset=["ticker", "Year"], keep="last").copy()
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

df[col_shares] = (
    df.groupby("ticker")[col_shares]
      .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
)

# Drop remaining missing shares
df = df.dropna(subset=[col_shares]).reset_index(drop=True)

# Final check
print("Rows:", len(df))
print("Missing shares:", df[col_shares].isna().sum())
print("Duplicate (ticker, Year):", df.duplicated(subset=["ticker", "Year"]).sum())
print(df.head())

Rows: 9386
Missing shares: 0
Duplicate (ticker, Year): 0
  ticker  Year  Outstanding Share (Mil. Shares)
0    A32  2019                        6800000.0
1    A32  2020                        6800000.0
2    A32  2021                        6800000.0
3    A32  2022                        6800000.0
4    A32  2023                        6800000.0


In [ ]:
# Export output
output_path = "annual_shares_outstanding_cleaned.xlsx"
df.to_excel(output_path, index=False)

print("✅ Cleaned file exported to:", output_path)

✅ Cleaned file exported to: annual_shares_outstanding_cleaned.xlsx


2. Balance sheets

In [ ]:
%pip install xlrd>=2.0.1
df = pd.read_excel("Balance_Sheets.xls")

# Standardize ticker
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

# Ensure Year numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

# Drop invalid identifiers
df = df.dropna(subset=["ticker", "Year"]).copy()

# Remove duplicate firm-year
df = df.drop_duplicates(subset=["ticker", "Year"], keep="last").copy()

# Sort as panel
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

# Convert columns to numeric values
financial_cols = df.columns.difference(["ticker", "Year"])

df[financial_cols] = df[financial_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

# Handle missing values
df[financial_cols] = (
    df.groupby("ticker")[financial_cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

# Drop rows with still-missing key balance sheet totals
key_cols = [
    "TOTAL ASSETS (Bn. VND)",
    "TOTAL LIABILITIES (Bn. VND)",
    "EQUITY (Bn. VND)"
]

existing_keys = [c for c in key_cols if c in df.columns]
df = df.dropna(subset=existing_keys).copy()

# Feature engineering
# Size proxy
if "TOTAL ASSETS (Bn. VND)" in df.columns:
    df["log_total_assets"] = np.log(df["TOTAL ASSETS (Bn. VND)"].clip(lower=1))

# Leverage
if all(c in df.columns for c in ["TOTAL LIABILITIES (Bn. VND)", "TOTAL ASSETS (Bn. VND)"]):
    df["leverage"] = df["TOTAL LIABILITIES (Bn. VND)"] / df["TOTAL ASSETS (Bn. VND)"]

# Book equity 
if "EQUITY (Bn. VND)" in df.columns:
    df["book_equity"] = df["EQUITY (Bn. VND)"]

# Final check 
print("Rows:", len(df))
print("Firms:", df["ticker"].nunique())
print("Missing values (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Export output
output_path = "Balance_Sheets_Cleaned.xlsx"
df.to_excel(output_path, index=False)

print("✅ Cleaned balance sheet file exported to:", output_path)

✅ Cleaned balance sheet file exported to: Balance_Sheets_Cleaned.xlsx


3. Cash Flows Statements

In [ ]:
df = pd.read_excel("Cash_Flows.xls")

# Standardize ticker
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

# Ensure Year numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

# Drop invalid firm-year identifiers
df = df.dropna(subset=["ticker", "Year"]).copy()

# Remove duplicate firm-year observations
df = df.drop_duplicates(subset=["ticker", "Year"], keep="last").copy()

# Sort as proper panel
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

# Convert columns to numeric values
financial_cols = df.columns.difference(["ticker", "Year"])

df[financial_cols] = df[financial_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

# Handle missing values
df[financial_cols] = (
    df.groupby("ticker")[financial_cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

# Drop rows where core cash flow items are missing
core_cf_cols = [
    "Net cash flow from operating activities",
    "Net cash flow from investing activities",
    "Net cash flow from financing activities"
]

existing_core = [c for c in core_cf_cols if c in df.columns]
df = df.dropna(subset=existing_core).copy()

# Feature engineering
# Operating cash flow proxy
if "Net cash flow from operating activities" in df.columns:
    df["operating_cf"] = df["Net cash flow from operating activities"]

# Investment intensity
if all(c in df.columns for c in [
    "Net cash flow from investing activities",
    "Net cash flow from operating activities"
]):
    df["investment_cf_ratio"] = (
        df["Net cash flow from investing activities"] /
        df["Net cash flow from operating activities"].replace(0, np.nan)
    )

# Financing dependence
if "Net cash flow from financing activities" in df.columns:
    df["financing_cf"] = df["Net cash flow from financing activities"]

# Final check
print("Rows:", len(df))
print("Firms:", df["ticker"].nunique())
print("Top missing columns:")
print(df.isna().sum().sort_values(ascending=False).head(10))

Rows: 9481
Firms: 1720
Top missing columns:
Dividends received                                     9436
_Increase/Decrease in receivables                      9330
Payment from reserves                                  9325
Profits from other activities                          9307
Net Cash Flows from Operating Activities before BIT    9307
_Increase/Decrease in payables                         9280
Profit/Loss from disposal of fixed assets              9079
Interest income and dividends                          8947
Payments for share repurchases                         7354
Finance lease principal payments                       7286
dtype: int64


In [ ]:
# Export output
output_path = "Cash_Flows_Cleaned.xlsx"
df.to_excel(output_path, index=False)

print("✅ Cleaned cash flow file exported to:", output_path)

✅ Cleaned cash flow file exported to: Cash_Flows_Cleaned.xlsx


4. Financial Ratios

In [ ]:
df = pd.read_excel("Financial_Ratios.xls")

# Standardize ticker
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

# Drop invalid firm-year identifiers
df = df.dropna(subset=["ticker", "Year"]).copy()

# Remove duplicate firm-year rows
df = df.drop_duplicates(subset=["ticker", "Year"], keep="last").copy()

# Sort as proper panel
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

# Convert to numerical values
ratio_cols = df.columns.difference(["ticker", "Year"])

df[ratio_cols] = df[ratio_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

# Handle extreme values
# Winsorize ratios within year to avoid outliers dominating ML
def winsorize_series(s, lower=0.01, upper=0.99):
    if s.notna().sum() < 10:
        return s
    return s.clip(s.quantile(lower), s.quantile(upper))

df[ratio_cols] = (
    df.groupby("Year")[ratio_cols]
      .transform(lambda x: winsorize_series(x))
)

# Handle missing values
df[ratio_cols] = (
    df.groupby("ticker")[ratio_cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

# Drop rows where key ratios are missing
key_ratio_candidates = [
    "ROA",
    "ROE",
    "Gross Margin",
    "Operating Margin",
    "Debt to Equity",
    "Current Ratio"
]

existing_keys = [c for c in key_ratio_candidates if c in df.columns]
df = df.dropna(subset=existing_keys).copy()

# Final check
print("Rows:", len(df))
print("Firms:", df["ticker"].nunique())
print("Remaining missing values (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

Rows: 9300
Firms: 1689
Remaining missing values (top 10):
Dividend yield (%)            1917
(ST+LT borrowings)/Equity      546
Interest Coverage              430
Inventory Turnover             324
Days Inventory Outstanding     324
BVPS (VND)                     176
P/B                            176
P/S                            165
EPS (VND)                       88
P/Cash Flow                     88
dtype: int64


In [ ]:
# Export output
output_path = "Financial_Ratios_Cleaned.xlsx"
df.to_excel(output_path, index=False)

print("✅ Cleaned financial ratios file exported to:", output_path)

✅ Cleaned financial ratios file exported to: Financial_Ratios_Cleaned.xlsx


5. Income Statements

In [ ]:
df = pd.read_excel("Income_Statement.xls")

# Standardize ticker
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

# Drop invalid firm-year identifiers
df = df.dropna(subset=["ticker", "Year"]).copy()

# Remove duplicate firm-year rows
df = df.drop_duplicates(subset=["ticker", "Year"], keep="last").copy()

# Sort as panel
df = df.sort_values(["ticker", "Year"]).reset_index(drop=True)

# Convert to numerical values
income_cols = df.columns.difference(["ticker", "Year"])

df[income_cols] = df[income_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

# Handle missing values
df[income_cols] = (
    df.groupby("ticker")[income_cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

# Drop rows where core income items are missing
core_income_cols = [
    "Net revenue",
    "Gross profit",
    "Operating profit",
    "Profit after tax"
]

existing_core = [c for c in core_income_cols if c in df.columns]
df = df.dropna(subset=existing_core).copy()

# Feature engineering
# Profitability proxy
if all(c in df.columns for c in ["Profit after tax", "Net revenue"]):
    df["net_profit_margin"] = (
        df["Profit after tax"] / df["Net revenue"].replace(0, np.nan)
    )

# Operating margin
if all(c in df.columns for c in ["Operating profit", "Net revenue"]):
    df["operating_margin"] = (
        df["Operating profit"] / df["Net revenue"].replace(0, np.nan)
    )

# Revenue growth (year-over-year)
if "Net revenue" in df.columns:
    df["revenue_growth"] = (
        df.groupby("ticker")["Net revenue"]
          .pct_change()
    )

# Final check
print("Rows:", len(df))
print("Firms:", df["ticker"].nunique())
print("Remaining missing values (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

Rows: 9478
Firms: 1720
Remaining missing values (top 10):
Net gain (loss) from trading of trading securities        9340
Net gain (loss) from disposal of investment securities    9310
Net Fee and Commission Income                             9304
Fees and Comission Expenses                               9304
Net Interest Income                                       9304
Fees and Comission Income                                 9304
Operating Profit before Provision                         9304
Provision for credit losses                               9304
Dividends received                                        9304
Total operating revenue                                   9304
dtype: int64


In [ ]:
# Export output
output_path = "Income_Statement_Cleaned.xlsx"
df.to_excel(output_path, index=False)

print("✅ Cleaned income statement file exported to:", output_path)

✅ Cleaned income statement file exported to: Income_Statement_Cleaned.xlsx


6. Stock Price

In [ ]:
df = pd.read_csv("Stock_data.csv")

# Rename Stock_code to ticker for consistency
df = df.rename(columns={"Stock_code": "ticker"})

# Clean ticker
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

# Parse date
date_col = "time"  

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Drop invalid rows
df = df.dropna(subset=["ticker", date_col]).copy()

# Sort properly
df = df.sort_values(["ticker", date_col]).reset_index(drop=True)

# Identify price column
price_candidates = ["adj_close", "Adj Close", "close", "Close", "price", "Price"]
price_col = next((c for c in price_candidates if c in df.columns), None)

if price_col is None:
    raise ValueError("❌ No price column found. Check your CSV.")

df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
df = df.dropna(subset=[price_col]).copy()

# Remove zero / negative prices
df = df[df[price_col] > 0].copy()

# Convert to monthly data
df["YearMonth"] = df[date_col].dt.to_period("M")

monthly_price = (
    df.groupby(["ticker", "YearMonth"])[price_col]
      .last()
      .reset_index()
)

monthly_price["YearMonth"] = monthly_price["YearMonth"].dt.to_timestamp()

# Compute monthly returns
monthly_price = monthly_price.sort_values(
    ["ticker", "YearMonth"]
).reset_index(drop=True)

monthly_price["return"] = (
    monthly_price.groupby("ticker")[price_col]
    .pct_change()
)

# Handle extreme returns
def winsorize(s, lower=0.01, upper=0.99):
    if s.notna().sum() < 20:
        return s
    return s.clip(s.quantile(lower), s.quantile(upper))

monthly_price["return"] = (
    monthly_price.groupby("YearMonth")["return"]
    .transform(winsorize)
)

# Drop first observation per stock
monthly_price = monthly_price.dropna(subset=["return"]).copy()

# Final check
print("Observations:", len(monthly_price))
print("Stocks:", monthly_price["ticker"].nunique())
print(monthly_price.head())

Observations: 121524
Stocks: 1670
  ticker  YearMonth  close    return
1    A32 2019-08-01  13.86  0.080281
2    A32 2019-09-01  15.59  0.124820
3    A32 2019-10-01  15.59  0.000000
4    A32 2019-11-01  16.17  0.037203
5    A32 2019-12-01  17.05  0.054422


In [ ]:
# Export output
output_path = "Stock_data_Monthly_Returns_Cleaned.xlsx"
monthly_price.to_excel(output_path, index=False)

print("✅ Cleaned stock return file exported to:", output_path)

✅ Cleaned stock return file exported to: Stock_data_Monthly_Returns_Cleaned.xlsx


7. Vietnam Government Bond Yield

In [ ]:
file_path = "Vietnam 10-Year Bond Yield Historical Data.csv"
df = pd.read_csv(file_path)

# Standardize column names
df.columns = [c.strip() for c in df.columns]
print(df.columns)

# Parse date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).copy()

# Clean yield column
df["yield"] = (
    df["Price"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

df["yield"] = pd.to_numeric(df["yield"], errors="coerce")

# Drop invalid rows
df = df.dropna(subset=["yield"]).copy()

# Convert to monthly data
df["YearMonth"] = df["Date"].dt.to_period("M")

monthly_yield = (
    df.groupby("YearMonth")["yield"]
      .last()
      .reset_index()
)

monthly_yield["YearMonth"] = monthly_yield["YearMonth"].dt.to_timestamp()

# Convert yield to monthly risk-free return
monthly_yield["rf"] = (monthly_yield["yield"] / 100) / 12

# Final check
print(monthly_yield.head())
print("Date range:", monthly_yield["YearMonth"].min(),
      "→", monthly_yield["YearMonth"].max())

Index(['Date', 'Price', 'Open', 'High', 'Low', 'Change %'], dtype='object')
   YearMonth  yield        rf
0 2020-02-01  2.856  0.002380
1 2020-03-01  3.517  0.002931
2 2020-04-01  3.050  0.002542
3 2020-05-01  3.146  0.002622
4 2020-06-01  3.033  0.002527
Date range: 2020-02-01 00:00:00 → 2025-12-01 00:00:00


In [ ]:
# Export output
output_path = "Vietnam_Risk_Free_Rate_Monthly.xlsx"
monthly_yield.to_excel(output_path, index=False)

print("✅ Cleaned risk-free rate file exported:", output_path)

✅ Cleaned risk-free rate file exported: Vietnam_Risk_Free_Rate_Monthly.xlsx


8. VNI Index

In [ ]:
file_path = "VN Index Historical Data.csv"
df = pd.read_csv(file_path)

# Standardize column names
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())

# Parse date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).copy()

# Clean index level
df["index_level"] = (
    df["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["index_level"] = pd.to_numeric(df["index_level"], errors="coerce")
df = df.dropna(subset=["index_level"]).copy()

# Remove invalid index levels
df = df[df["index_level"] > 0].copy()

# Convert to monthly data
df["YearMonth"] = df["Date"].dt.to_period("M")

monthly_index = (
    df.groupby("YearMonth")["index_level"]
      .last()
      .reset_index()
)

monthly_index["YearMonth"] = monthly_index["YearMonth"].dt.to_timestamp()

# Compute monthly market returns
monthly_index = monthly_index.sort_values("YearMonth").reset_index(drop=True)

monthly_index["market_return"] = (
    monthly_index["index_level"].pct_change()
)

# Drop first observation
monthly_index = monthly_index.dropna(subset=["market_return"]).copy()

# Winsorize extreme returns to enhance data quality
def winsorize(s, lower=0.01, upper=0.99):
    if s.notna().sum() < 20:
        return s
    return s.clip(s.quantile(lower), s.quantile(upper))

monthly_index["market_return"] = winsorize(monthly_index["market_return"])

# Final check
print("Observations:", len(monthly_index))
print("Date range:",
      monthly_index["YearMonth"].min(),
      "→",
      monthly_index["YearMonth"].max())
print(monthly_index.head())

Columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
Observations: 71
Date range: 2020-02-01 00:00:00 → 2025-12-01 00:00:00
   YearMonth  index_level  market_return
1 2020-02-01       882.19      -0.058113
2 2020-03-01       662.53      -0.155822
3 2020-04-01       769.11       0.135052
4 2020-05-01       864.47       0.123987
5 2020-06-01       825.11      -0.045531


In [ ]:
# Export output
output_path = "VN_Index_Monthly_Market_Return_Cleaned.xlsx"
monthly_index.to_excel(output_path, index=False)

print("✅ Cleaned market return file exported:", output_path)

✅ Cleaned market return file exported: VN_Index_Monthly_Market_Return_Cleaned.xlsx


II. Build the master research panel

In [ ]:
# Load datasets
returns = pd.read_excel("Stock_data_Monthly_Returns_Cleaned.xlsx")
shares  = pd.read_excel("annual_shares_outstanding_cleaned.xlsx")
bs      = pd.read_excel("Balance_Sheets_Cleaned.xlsx")
is_df   = pd.read_excel("Income_Statement_Cleaned.xlsx")
cf      = pd.read_excel("Cash_Flows_Cleaned.xlsx")
ratios  = pd.read_excel("Financial_Ratios_Cleaned.xlsx")
mkt = pd.read_excel("VN_Index_Monthly_Market_Return_Cleaned.xlsx")
rf  = pd.read_excel("Vietnam_Risk_Free_Rate_Monthly.xlsx")

# Standardize keys
for df in [returns, shares, bs, is_df, cf, ratios]:
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

# Convert YearMonth to Year
returns["YearMonth"] = pd.to_datetime(returns["YearMonth"], errors="coerce")
returns["Year"] = returns["YearMonth"].dt.year
mkt["YearMonth"] = pd.to_datetime(mkt["YearMonth"], errors="coerce")
rf["YearMonth"]  = pd.to_datetime(rf["YearMonth"], errors="coerce")

# Keep only needed columns
mkt = mkt[["YearMonth", "market_return"]].dropna()
rf  = rf[["YearMonth", "rf"]].dropna()

# Merge accounting data
acct = (
    shares
    .merge(bs,     on=["ticker", "Year"], how="left")
    .merge(is_df,  on=["ticker", "Year"], how="left")
    .merge(cf,     on=["ticker", "Year"], how="left")
    .merge(ratios, on=["ticker", "Year"], how="left")
)

# Lag accounting variables by 1 year
acct_lagged = acct.copy()
acct_lagged["Year"] = acct_lagged["Year"] + 1

# Merge with monthly returns
panel = returns.merge(
    acct_lagged,
    on=["ticker", "Year"],
    how="inner"
)

# Merge market returns and risk-free rate into panel
panel["YearMonth"] = pd.to_datetime(panel["YearMonth"], errors="coerce")
panel = panel.merge(mkt, on="YearMonth", how="left")
panel = panel.merge(rf,  on="YearMonth", how="left")

# Compute excess returns
panel["excess_return"] = panel["return"] - panel["rf"]
panel["mkt_excess"]    = panel["market_return"] - panel["rf"]

# Feature engineering
# Market capitalization (size)
if "Outstanding Share (Mil. Shares)" in panel.columns:
    panel["market_cap"] = (
        panel["Outstanding Share (Mil. Shares)"] * panel.iloc[:, panel.columns.get_loc("price")]
    )
    panel["log_mktcap"] = np.log(panel["market_cap"].clip(lower=1))

# Book-to-market
if "book_equity" in panel.columns:
    panel["bm"] = panel["book_equity"] / panel["market_cap"].replace(0, np.nan)

# Momentum (12–2)
panel = panel.sort_values(["ticker", "YearMonth"])

panel["mom_12_2"] = (
    panel.groupby("ticker")["return"]
    .rolling(12)
    .apply(lambda x: (1 + x[:-1]).prod() - 1, raw=False)
    .reset_index(level=0, drop=True)
)

# Volatility (12-month rolling)
panel["vol_12"] = (
    panel.groupby("ticker")["return"]
    .rolling(12)
    .std()
    .reset_index(level=0, drop=True)
)

# Drop rows without dependent variable
panel = panel.dropna(subset=["return"]).copy()
panel = panel.dropna(subset=["market_return", "rf", "excess_return", "mkt_excess"]).copy()

# Final check
print("Observations:", len(panel))
print("Stocks:", panel["ticker"].nunique())
print("Date range:", panel["YearMonth"].min(), "→", panel["YearMonth"].max())

Observations: 104507
Stocks: 1632
Date range: 2020-02-01 00:00:00 → 2025-12-01 00:00:00


In [ ]:
# Export output
panel.to_csv("Master_Research_Panel.csv", index=False)

print("✅ Exported file: Master_Research_Panel.csv")

✅ Exported file: Master_Research_Panel.csv


Final data cleaning and preparation

In [ ]:
# Check for the required data sources in the master reseach panel
file_path = "Master_Research_Panel.csv"
df = pd.read_csv(file_path)

print("Columns:", df.columns.tolist())

Columns: ['ticker', 'YearMonth', 'close', 'return', 'Year', 'Outstanding Share (Mil. Shares)_x', 'CURRENT ASSETS (Bn. VND)', 'Cash and cash equivalents (Bn. VND)', 'Accounts receivable (Bn. VND)', 'Net Inventories', 'Other current assets', 'LONG-TERM ASSETS (Bn. VND)', 'Fixed assets (Bn. VND)', 'Long-term investments (Bn. VND)', 'Other non-current assets', 'TOTAL ASSETS (Bn. VND)', 'LIABILITIES (Bn. VND)', 'Current liabilities (Bn. VND)', 'Long-term liabilities (Bn. VND)', "OWNER'S EQUITY(Bn.VND)", 'Capital and reserves (Bn. VND)', 'Undistributed earnings (Bn. VND)', 'Budget sources and other funds', 'TOTAL RESOURCES (Bn. VND)', 'Prepayments to suppliers (Bn. VND)', 'Inventories, Net (Bn. VND)', 'Investment and development funds (Bn. VND)', 'Common shares (Bn. VND)', 'Paid-in capital (Bn. VND)', 'Long-term borrowings (Bn. VND)', 'Advances from customers (Bn. VND)', 'Short-term borrowings (Bn. VND)', 'Long-term prepayments (Bn. VND)', 'Other long-term assets (Bn. VND)', 'Short-term inve

Keep the necessary columns

In [ ]:
panel = pd.read_csv("Master_Research_Panel.csv")

keep_cols = [
    # identifiers
    "ticker", "YearMonth", "Year",

    # price/returns (dependent variables)
    "close", "return", "excess_return",

    # market + risk-free (CAPM / FF need these)
    "market_return", "rf", "mkt_excess",

    # time-series predictors
    "mom_12_2", "vol_12",

    # size proxy (keep shares; keep market cap if you later recompute)
    "Outstanding Share (Mil. Shares)_x",
]

# Keep financial ratios columns
ratio_cols = [
    "lengthReport",
    "(ST+LT borrowings)/Equity",
    "Debt/Equity",
    "Fixed Asset-To-Equity",
    "Owners' Equity/Charter Capital",
    "Asset Turnover",
    "Fixed Asset Turnover",
    "Days Sales Outstanding",
    "Days Inventory Outstanding",
    "Days Payable Outstanding",
    "Cash Cycle",
    "Inventory Turnover",
    "EBIT Margin (%)",
    "Gross Profit Margin (%)",
    "Net Profit Margin (%)",
    "ROE (%)",
    "ROIC (%)",
    "ROA (%)",
    "EBITDA (Bn. VND)",
    "EBIT (Bn. VND)",
    "Dividend yield (%)",
    "Current Ratio",
    "Cash Ratio",
    "Quick Ratio",
    "Financial Leverage",
    "Market Capital (Bn. VND)",
    "Outstanding Share (Mil. Shares)_y",
    "P/E",
    "P/B",
    "P/S",
    "P/Cash Flow",
    "EPS (VND)",
    "BVPS (VND)",
    "EV/EBITDA",
    "Interest Coverage"
]

# Combine
keep_cols_final = [c for c in (keep_cols + ratio_cols) if c in panel.columns]

panel_reduced = panel[keep_cols_final].copy()

# Drop duplicate columns
if "Outstanding Share (Mil. Shares)_x" in panel_reduced.columns and "Outstanding Share (Mil. Shares)_y" in panel_reduced.columns:
    panel_reduced = panel_reduced.drop(columns=["Outstanding Share (Mil. Shares)_y"])

# Export output
panel_reduced.to_csv("Master_Research_Panel_Reduced.csv", index=False)

print("✅ Saved:", "Master_Research_Panel_Reduced.csv")
print("Final columns:", panel_reduced.columns.tolist())
print("Shape:", panel_reduced.shape)

✅ Saved: Master_Research_Panel_Reduced.csv
Final columns: ['ticker', 'YearMonth', 'Year', 'close', 'return', 'excess_return', 'market_return', 'rf', 'mkt_excess', 'mom_12_2', 'vol_12', 'Outstanding Share (Mil. Shares)_x', 'lengthReport', '(ST+LT borrowings)/Equity', 'Debt/Equity', 'Fixed Asset-To-Equity', "Owners' Equity/Charter Capital", 'Asset Turnover', 'Fixed Asset Turnover', 'Days Sales Outstanding', 'Days Inventory Outstanding', 'Days Payable Outstanding', 'Cash Cycle', 'Inventory Turnover', 'EBIT Margin (%)', 'Gross Profit Margin (%)', 'Net Profit Margin (%)', 'ROE (%)', 'ROIC (%)', 'ROA (%)', 'EBITDA (Bn. VND)', 'EBIT (Bn. VND)', 'Dividend yield (%)', 'Current Ratio', 'Cash Ratio', 'Quick Ratio', 'Financial Leverage', 'Market Capital (Bn. VND)', 'P/E', 'P/B', 'P/S', 'P/Cash Flow', 'EPS (VND)', 'BVPS (VND)', 'EV/EBITDA', 'Interest Coverage']
Shape: (104507, 46)


Group by Sector

In [ ]:
panel_path = 'Master_Research_Panel_Reduced.csv'
sector_path = 'Stock_List.xlsx'

def load_and_prep_research_data(panel_path, sector_path):
    print("Step 1: Loading datasets...")
    panel_df = pd.read_csv(panel_path)
    panel_df['YearMonth'] = pd.to_datetime(panel_df['YearMonth'])
    try:
        stock_list = pd.read_excel(sector_path)
    except Exception as e:
        print(f"Error loading excel: {e}")
        return None

    # Standardize 'ticker' column name across both dataframes
    if 'Stock_code' in stock_list.columns:
        stock_list.rename(columns={'Stock_code': 'ticker'}, inplace=True)
    
    # Clean whitespace and extract unique mapping
    stock_list.columns = [c.strip() for c in stock_list.columns]
    sector_map = stock_list[['ticker', 'Sector']].drop_duplicates()
    
    print("Step 2: Merging and Cleaning...")
    
    # Merge ticker with sector
    df = pd.merge(panel_df, sector_map, on='ticker', how='inner')
    
    # Drop rows with critical missing values
    critical_cols = ['excess_return', 'mkt_excess']
    df.dropna(subset=critical_cols, inplace=True)
    
    # Sort by ticker and time to maintain panel structure
    df = df.sort_values(['ticker', 'YearMonth']).reset_index(drop=True)
    
    # Export output
    output_filename = 'Master_Panel.csv'
    df.to_csv(output_filename, index=False)
    
    print(f"Success: Merged data contains {df.shape[0]} observations.")
    print(f"Result exported to: {output_filename}")
    
    return df